In [2]:
import os
import json
import re
import pkg_resources
from pathlib import Path

# --- Step 1: Define project root and files to scan ---
project_root = Path(".")  # current folder
files_to_scan = [
    "data_splitting.py",
    "evaluate.py",
    "main.ipynb",
    "model_train.ipynb",
    "preprocess.py",
    "similarity_across_splits.py",
    "utils_config.py",
    "config.yaml"
]

# --- Step 2: Regex to extract imports ---
import_pattern = re.compile(r'^\s*(?:import|from)\s+([^\s\.]+)', re.MULTILINE)

imports = set()
special_cases = {"sklearn": "scikit-learn"}

# --- Step 3: Scan each file ---
for filename in files_to_scan:
    path = project_root / filename
    if not path.exists():
        print(f"⚠️ Skipping missing file: {filename}")
        continue

    if filename.endswith(".ipynb"):
        with open(path, "r", encoding="utf-8") as f:
            notebook = json.load(f)
        code_cells = [cell["source"] for cell in notebook["cells"] if cell["cell_type"] == "code"]
        for cell_source in code_cells:
            code_text = "".join(cell_source) if isinstance(cell_source, list) else cell_source
            imports.update(import_pattern.findall(code_text))
    elif filename.endswith(".py"):
        with open(path, "r", encoding="utf-8") as f:
            code_text = f.read()
        imports.update(import_pattern.findall(code_text))

# --- Step 4: Resolve installed versions ---
requirements = []
special_cases = {"sklearn": "scikit-learn", "yaml": "pyyaml"}

# standard library modules to skip
stdlib_skip = {
    "os", "sys", "math", "json", "re", "pathlib", "collections", "itertools", "warnings"
}

# your local project modules to skip
local_skip = {
    "preprocess", "data_splitting", "model_train", "evaluate",
    "similarity_across_splits", "utils_config"
}

for package in sorted(imports):
    if package in stdlib_skip or package in local_skip:
        continue  # skip stdlib & local files

    lookup_package = special_cases.get(package, package)
    try:
        dist = pkg_resources.get_distribution(lookup_package)
        requirements.append(f"{dist.key}=={dist.version}")
    except Exception:
        print(f"⚠️ Could not find version for package: {package} → adding without version")
        requirements.append(lookup_package)  # add at least the name


# --- Step 5: Save requirements.txt ---
with open("requirements.txt", "w") as f:
    f.write("\n".join(requirements))
print("✅ requirements.txt created!")

# --- Step 6: Save folder structure ---
def list_files(startpath):
    lines = []
    for root, dirs, files in os.walk(startpath):
        level = root.replace(str(startpath), "").count(os.sep)
        indent = " " * 4 * level
        lines.append(f"{indent}{os.path.basename(root)}/")
        subindent = " " * 4 * (level + 1)
        for f in files:
            lines.append(f"{subindent}{f}")
    return "\n".join(lines)

with open("project_structure.txt", "w") as f:
    f.write(list_files(project_root))

print("✅ project_structure.txt created!")


⚠️ Skipping missing file: model_train.ipynb
✅ requirements.txt created!
✅ project_structure.txt created!
